# MAMMAL DTI Smoke Test

Reproduces the IBM model card's canonical inference call locally. If this cell runs and prints a reasonable pKd, your env + GPU + weights are working and you're ready to run the full pipeline.

Expected behavior:
- Cell 1 imports without error (validates `biomed-multi-alignment[examples]` install)
- Cell 2 loads the model (~1.8 GB download on first run; cached after)
- Cell 3 prints a pKd for galantamine vs CHRNA7 — should be a finite float roughly in [4, 8]
- Cell 4 prints a pKd for the model card's reference (target/drug pair from HF README)

In [ ]:
import torch
from fuse.data.tokenizers.modular_tokenizer.op import ModularTokenizerOp
from mammal.examples.dti_bindingdb_kd.task import DtiBindingdbKdTask
from mammal.model import Mammal

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

In [ ]:
MODEL_ID = "ibm/biomed.omics.bl.sm.ma-ted-458m.dti_bindingdb_pkd"
NORM_Y_MEAN = 5.79384684128215
NORM_Y_STD = 1.33808027428196

model = Mammal.from_pretrained(MODEL_ID)
model.eval()
if torch.cuda.is_available():
    model.to("cuda")

tokenizer_op = ModularTokenizerOp.from_pretrained(MODEL_ID)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded. Parameters: {n_params/1e6:.1f}M. Device: {model.device}.")

## Test 1 — Galantamine vs CHRNA7

Galantamine is one of the few approved CHRNA7 (alpha-7 nAChR) modulators (positive allosteric modulator). The DTI head should produce a meaningful pKd, though absolute calibration on allosteric modulators is known to be soft.

In [ ]:
from mammal_repurposing.fetchers.uniprot import fetch_sequence
from mammal_repurposing.fetchers.pubchem import fetch_smiles

chrna7 = fetch_sequence("P36544")
galantamine = fetch_smiles("galantamine")
print(f"CHRNA7 length: {chrna7['length']} AA")
print(f"Galantamine SMILES: {galantamine['smiles']}")

sample = {"target_seq": chrna7['sequence'], "drug_seq": galantamine['smiles']}
sample = DtiBindingdbKdTask.data_preprocessing(
    sample_dict=sample,
    tokenizer_op=tokenizer_op,
    target_sequence_key="target_seq",
    drug_sequence_key="drug_seq",
    norm_y_mean=None,
    norm_y_std=None,
    device=model.device,
)
batch_dict = model.forward_encoder_only([sample])
batch_dict = DtiBindingdbKdTask.process_model_output(
    batch_dict,
    scalars_preds_processed_key="model.out.dti_bindingdb_kd",
    norm_y_mean=NORM_Y_MEAN,
    norm_y_std=NORM_Y_STD,
)
pkd = float(batch_dict["model.out.dti_bindingdb_kd"][0])
print(f"\nPredicted pKd(galantamine, CHRNA7) = {pkd:.3f}")
print(f"(Expected: finite float roughly in [4, 8] for an approved CNS drug.)")

## Test 2 — HF README reference pair

Same exact target/drug pair as the model card's example, to confirm we get a sensible number against IBM's reference call.

In [ ]:
# Reference inputs from the HF model card README
target_seq = "NLMKRCTRGFRKLGKCTTLEEEKCKTLYPRGQCTCSDSKMNTHSCDCKSC"
drug_seq = "CC(=O)NCCC1=CNc2c1cc(OC)cc2"

sample = {"target_seq": target_seq, "drug_seq": drug_seq}
sample = DtiBindingdbKdTask.data_preprocessing(
    sample_dict=sample,
    tokenizer_op=tokenizer_op,
    target_sequence_key="target_seq",
    drug_sequence_key="drug_seq",
    norm_y_mean=None,
    norm_y_std=None,
    device=model.device,
)
batch_dict = model.forward_encoder_only([sample])
batch_dict = DtiBindingdbKdTask.process_model_output(
    batch_dict,
    scalars_preds_processed_key="model.out.dti_bindingdb_kd",
    norm_y_mean=NORM_Y_MEAN,
    norm_y_std=NORM_Y_STD,
)
pkd_ref = float(batch_dict["model.out.dti_bindingdb_kd"][0])
print(f"Reference pair pKd = {pkd_ref:.3f}")

## Test 3 — Batch inference

Quick check that batched inference works and gives consistent results with single-pair inference.

In [ ]:
from mammal_repurposing.scoring.dti import score_batch, score_pair
from mammal_repurposing.scoring.model_loader import load_dti_model

# Use the cached loader so we don't reload
m, tok = load_dti_model()

donepezil = fetch_smiles("donepezil")
ache = fetch_sequence("P22303")

pkd_single = score_pair(m, tok, ache['sequence'], donepezil['smiles'])
pkd_batch = score_batch(m, tok, [
    (chrna7['sequence'], galantamine['smiles']),
    (ache['sequence'], donepezil['smiles']),
])
print(f"single pKd(donepezil, ACHE) = {pkd_single:.3f}")
print(f"batch  pKd(galantamine, CHRNA7) = {pkd_batch[0]:.3f}")
print(f"batch  pKd(donepezil, ACHE) = {pkd_batch[1]:.3f}")
print(f"single/batch consistency: {abs(pkd_single - pkd_batch[1]) < 1e-3}")